# SCTEC - Projeto Final — Módulo 01 | Machine Learning: Modelo Preditivo
**DESENVOLVIMENTO DE IA PARA ANÁLISE PREDITIVA**

---

## Descrição do notebook

Este material implementa um  Pipeline  Preditivo  completo aplicado à Indústria 4.0. 

●  O  algorítmo propõe uma solução preditiva para um  parque  fabril  monitorado  por  sensores, prevendo  quebras mecânicas  nos  equipamentos,  para  evitar paradas  na  linha  de  produção.  A  variável alvo  do  projeto  é  binária:  falha_maquina  =  1 (quando  há  uma  avaria  detectada)  e  falha_maquina  =  0 (funcionamento normal). 
Na solução são aplicados os conceitos de:

* **Divisão treino/teste com estratificação:** por que `stratify=y` importa quando o alvo é desbalanceado;
* **Balanceamento de classes:** SMOTE aplicado **apenas no treino**, para não vazar informação para o teste;
* **Escalonamento seletivo:** por que o KNN precisa de `StandardScaler` e a Árvore de Decisão não;
* **Ajuste de hiperparâmetros:** `n_neighbors` no KNN e `max_depth` na Árvore, e como cada um controla o overfitting;
* **Avaliação crítica de acurácia:** por que "acerto alto" pode significar pouco quando o alvo é raro.

💡 **Sobre a base:** ela foi criada com problemas propositais — nulos, duplicatas, colunas de vazamento de dados e desbalanceamento de classe — para proprorcionar as capacidades de avaliação, diagnóstico e decisão, não só sintaxe.

⚠️ **Atenção:** Trata-se de um projeto acadêmico visando ao atendimento às competências necessárias para a conclusão com aproveitamento no curso de "Desenvolvimento de IA para Análise Preditiva, do projeto SCTEC.

⚠️ **Atenção:** A cosntrução desse projeto observou a estruturação manual da hierarquia de pastas, sendo pois, necessária a sua replicação, para o seu correto funcionamento.

## FASE 0: Configuração de Ambiente e Repositório Git

### 0.1 - Ambiente e importações

Na maioria dos ambientes de desenvolvimento, as bibliotecas abaixo já está disponível. A exceção costuma ser o `imbalanced-learn` (SMOTE):
O comandos abaixo deveram ser executados no terminal, visando à instalação da biblioteca.
```
pip install imbalanced-learn
```
Para rodar localmente:
```
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn
```

In [ ]:
# Bibliotecas de terceiros
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from imblearn.over_sampling import SMOTE

# Módulos da biblioteca padrão do Python
import os
import re
import json
from datetime import datetime, timedelta
from IPython.display import display

pd.set_option("display.width", 120)
sns.set_style("whitegrid")

print("Ambiente pronto!")

## FASE 1: Análise Exploratória de Dados (EDA)

<h5 style="color: #7ac1f4;">
RF01 – Carregar o Dataset de Vendas
</h5>

In [ ]:
# Carregar o CSV
df_bruto = pd.read_csv('../data/manutencao_preditiva.csv')

<h5 style="color: #7ac1f4;">
RF02 – Inspecionar e Descrever os Dados
</h5>

In [ ]:
# Função para inspecionar os dados
def inspecionar_dados(df):
    """Exibe as informações estruturais do DataFrame com tabelas renderizadas."""
    print("=== INSPEÇÃO INICIAL DO DATASET ===")
    
    # 1. Shape
    print(f'Shape: O dataframe tem [{df.shape[0]}] linhas e [{df.shape[1]}] colunas.\n')
    
    # 2. As 5 primeiras linhas (renderizadas)
    print("Primeiras 5 linhas:")
    display(df.head()) 
    
    # 3. As 5 últimas linhas (renderizadas)
    print("\nÚltimas 5 linhas:")
    display(df.tail())
    
    # 4. Tipos das colunas
    print("\nTipos das colunas:")
    display(df.dtypes.to_frame(name="Tipo de Dado"))
    
    # 5. Valores nulos
    print("\nValores nulos por coluna:")
    display(df.isnull().sum().to_frame(name="Qtd Nulos"))

In [ ]:
# Chamada da função
inspecionar_dados(df_bruto)

In [ ]:
df_bruto.info()  # Exibe informações detalhadas sobre o DataFrame
df_bruto.describe()  # Exibe estatísticas descritivas do DataFrame

<h5 style="color: #7ac1f4;">
RF03 – 3 gráficos analíticos obrigatórios
</h5>

In [ ]:
# Configuração da figura com 3 subplots em 1 linha
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -------------------------------------------------------------------------
# GRÁFICO 1: Histograma de Distribuição por Classe (Torque em Nm)
# -------------------------------------------------------------------------
sns.histplot(
    data=df_bruto, 
    x="torque_nm", 
    hue="falha_maquina", 
    kde=True, 
    ax=axes[0], 
    palette=["#1C7293", "#D64550"]
)
axes[0].set_title("Distribuição do Torque (Nm) por Classe de Falha")
axes[0].set_xlabel("Torque (Nm)")
axes[0].set_ylabel("Frequência")

# -------------------------------------------------------------------------
# GRÁFICO 2: Gráfico de Barras do Desbalanceamento da Variável Alvo
# -------------------------------------------------------------------------
contagem = df_bruto["falha_maquina"].value_counts().sort_index()
axes[1].bar(["Sem Falha (0)", "Com Falha (1)"], contagem.values, color=["#1C7293", "#D64550"])

# Adiciona os rótulos de contagem e percentual sobre cada barra
for i, v in enumerate(contagem.values):
    percentual = (v / contagem.sum()) * 100
    axes[1].text(i, v + 150, f"{v}\n({percentual:.1f}%)", ha="center", fontsize=10, fontweight='bold')

axes[1].set_title("Desbalanceamento da Variável Alvo (falha_maquina)")
axes[1].set_ylabel("Quantidade de Equipamentos")
axes[1].set_ylim(0, max(contagem.values) * 1.15) # Ajusta limite Y para caber o texto

# -------------------------------------------------------------------------
# GRÁFICO 3: Mapa de Calor (Heatmap) da Correlação de Pearson
# -------------------------------------------------------------------------
cols_correlacao = [
    "temperatura_ar_k", "temperatura_processo_k", "velocidade_rotacao_rpm", 
    "torque_nm", "desgaste_ferramenta_min", "falha_maquina"
]

corr_matrix = df_bruto[cols_correlacao].corr(method="pearson")

sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="coolwarm", 
    center=0, 
    vmin=-1, 
    vmax=1, 
    ax=axes[2]
)
axes[2].set_title("Matriz de Correlação de Pearson")

# Ajuste fino de layout
plt.tight_layout()
plt.show()


:
### ✳️ Decisão  técnica - 01
A escolha da coluna torque_nm para o histograma da FASE 1 não foi arbitrária. Ela foi definida com base em duas análises estatísticas prévias realizadas na Fase 1 (a inspeção do info() e, principalmente, a análise descritiva com o describe()).

1. Maior Dispersão e Variabilidade Estatística no describe()
Na análise descritiva dos dados numéricos:

    - Variáveis como temperatura_ar_k e temperatura_processo_k possuem amplitude e desvio padrão relativamente baixos e muito concentrados em torno da média (pouca variação relativa).
    - O torque_nm, por outro lado, apresenta uma alta amplitude e variabilidade significativa (variando desde valores próximos de 3.8 Nm até mais de 76.6 Nm), tornando-se um candidato ideal para observar como a variação de um parâmetro operacional contínuo se distribui em relação às falhas. 
    - Distribuição Normal Natural: A igualdade entre média e mediana no describe() indica um formato de curva em sino perfeito no gráfico.Picos Extremos: O max em $76.60$ isola os eventos de esforço elevado, permitindo visualizar onde ocorrem as falhas por sobrecarga nas caudas do gráfico.
    - Maior dispersão Proporcional (CV = 25%): Entre as variáveis físicas operacionais, o torque é a que possui a oscilação mais expressiva em torno da média. 
    

1. Em suma, observou-se os seguintes aspectos anteriores para a escolha:

    - Maior Dispersão Proporcional (CV = 25%): Entre as variáveis físicas operacionais, o torque é a que possui a oscilação mais expressiva em torno da média;

    - Distribuição Normal Natural: A igualdade entre média e mediana no describe() indica um formato de curva em sino perfeito no gráfico;

    - Picos Extremos: O max em $76.60$ isola os eventos de esforço elevado, permitindo visualizar onde ocorrem as falhas por sobrecarga nas caudas do gráfico.

### 📊 Análise Interpretativa do Diagnóstico de EDA

A partir da inspeção das estatísticas e da visualização dos gráficos analíticos, identificou-se os seguintes padrões para o direcionamento do projeto:

1. **Distribuição do Torque e Sobrecarga Mecânica (Gráfico 1):**
   * A operação normal (Classe 0) segue uma distribuição normal centrada em aproximadamente 40 Nm. 
   * As falhas (Classe 1) concentram-se visivelmente nos extremos da distribuição, sobretudo em valores elevados de torque (> 55 Nm), indicando que episódios de sobrecarga mecânica são fortes desencadeadores de paradas não planejadas.

2. **Severo Desbalanceamento de Classes (Gráfico 2):**
   * A variável alvo `falha_maquina` possui 9.661 registros normais (96.6%) e apenas 339 falhas (3.4%).
   * *Orientação:* É fundamental aplicar reamostragem via **SMOTE** no conjunto de treino (FASE 4) para evitar viés em favor da classe majoritária.

3. **Matriz de Correlação de Pearson (Gráfico 3):**
   * **Principais Preditores:** O `torque_nm` (r = 0.19) e o `desgaste_ferramenta_min` (r = 0.11) apresentam as maiores correlações diretas com a ocorrência de falhas.
   * **Fundamentação para Feature Engineering:** A forte correlação negativa entre `velocidade_rotacao_rpm` e `torque_nm` (r = -0.88) reflete a dinâmica física dos motores elétricos, respaldando a criação do atributo de **potência mecânica** na FASE 3.
   * **Multicolinearidade Térmica:** A correlação de r = 0.88 entre `temperatura_ar_k` e `temperatura_processo_k` revela redundância de informação entre a temperatura ambiente e a do processo.

## FASE 2: Limpeza e Tratamento de Dados (Data Prep)

<h5 style="color: #7ac1f4;">
RF04 – Identificar duplicatas, nulos e ausentes
</h5>

In [ ]:
# 1. Trabalhar a partir do dataset original 'df_bruto'
df_tratado = df_bruto.copy()

# 2. Quantidade de linhas antes da remoção de duplicados
linhas_antes = len(df_tratado)

# 3. Remoção de duplicados com base na coluna 'id_produto'
df_tratado = df_tratado.drop_duplicates(subset=["id_produto"], keep="first").reset_index(drop=True)

# 4. Quantidade de linhas após a remoção de duplicados
linhas_depois = len(df_tratado)

print("Tratamento de duplicados concluído:\n")
print(f"Linhas antes: {linhas_antes} | depois: {linhas_depois} | removidas: {linhas_antes - linhas_depois}")

In [ ]:
# 5. Identificação de dados ausentes antes do tratamento

print("Valores ausentes por coluna:")
display(df_tratado.isna().sum().to_frame("quantidade"))

print("\nAssimetria (skew):")
print("temperatura_ar_k:", round(df_tratado["temperatura_ar_k"].skew(), 3))
print("temperatura_processo_k:", round(df_tratado["temperatura_processo_k"].skew(), 3))
print("velocidade_rotacao_rpm:", round(df_tratado["velocidade_rotacao_rpm"].skew(), 3))
print("torque_nm:", round(df_tratado["torque_nm"].skew(), 3))

In [ ]:
# 6. Imputação de dados ausentes utilizando a MEDIANA

cols_com_nulos = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
]

for col in cols_com_nulos:
  if col in df_tratado.columns and df_tratado[col].isnull().sum() > 0:
    mediana_val = df_tratado[col].median()
    df_tratado[col] = df_tratado[col].fillna(mediana_val)

print("\nValidação após imputação (deve retornar 0 para todas as colunas):")
print(df_tratado[cols_com_nulos].isnull().sum())

In [ ]:
# 7. Visualização de Outliers via Boxplots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Boxplots das Variáveis Explicativas Contínuas (Identificação de Outliers)",
    fontsize=16,
    fontweight="bold",
)

sns.boxplot(ax=axes[0, 0], y=df_tratado["temperatura_ar_k"], color="#3498db")
axes[0, 0].set_title("Temperatura do Ar (K)")

sns.boxplot(ax=axes[0, 1], y=df_tratado["temperatura_processo_k"], color="#2ecc71")
axes[0, 1].set_title("Temperatura do Processo (K)")

sns.boxplot(ax=axes[1, 0], y=df_tratado["velocidade_rotacao_rpm"], color="#e67e22")
axes[1, 0].set_title("Velocidade de Rotação (RPM)")

sns.boxplot(ax=axes[1, 1], y=df_tratado["torque_nm"], color="#e74c3c")
axes[1, 1].set_title("Torque (Nm)")

plt.tight_layout()
plt.show()

In [ ]:
# 7. Contabilizando de Outliersa usando a regra do Intervalo Interquartil (IQR)

# Comentário pessoal: Nesse ponto, estou utilizando a regra do Intervalo Interquartil (IQR) para identificar e contabilizar os outliers nas variáveis contínuas do dataset. O objetivo é entender melhor a distribuição dos dados e identificar possíveis pontos fora do padrão que possam influenciar negativamente na análise ou modelagem preditiva e verificar, pela quantidade, o impacto desses pontos na base de dados e a viabilidade de remoção ou tratamento.

# Lista das colunas operacionais contínuas
colunas_analise = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
]

print("--- Contagem de Outliers por Coluna (Regra IQR) ---")

for col in colunas_analise:
  # Calculando o 1º quartil (25%) e 3º quartil (75%)
  Q1 = df_tratado[col].quantile(0.25)
  Q3 = df_tratado[col].quantile(0.75)

  # Intervalo Interquartil
  IQR = Q3 - Q1

  # Limites inferior e superior
  limite_inferior = Q1 - 1.5 * IQR
  limite_superior = Q3 + 1.5 * IQR

  # Identificando as linhas fora dos limites
  outliers_inf = df_tratado[df_tratado[col] < limite_inferior]
  outliers_sup = df_tratado[df_tratado[col] > limite_superior]

  total_outliers = len(outliers_inf) + len(outliers_sup)
  percentual = (total_outliers / len(df_tratado)) * 100

  print(f"\nVariável: {col}")
  print(f"  • Limite Inferior: {limite_inferior:.2f} | Limite Superior: {limite_superior:.2f}")
  print(f"  • Outliers Inferiores: {len(outliers_inf)}")
  print(f"  • Outliers Superiores: {len(outliers_sup)}")
  print(f"  • Total de Outliers: {total_outliers} ({percentual:.2f}% dos dados)")

### ✳️ Decisão  técnica - 02 

Tratamento de Dados Ausentes e Análise de Outliers (Card 2.1) - A partir do conjunto original `df_bruto`, realizou-se a cópia de trabalho, a remoção de duplicados e o tratamento de nulos.

#### Justificativa Técnica para Imputação via Mediana
Para o preenchimento dos valores nulos identificados nas variáveis operacionais (`temperatura_ar_k`, `temperatura_processo_k`, `velocidade_rotacao_rpm` e `torque_nm`), optou-se pela **Mediana** em detrimento da Média pelos seguintes motivos estatísticos:

1. **Robustez a Valores Extremos (Outliers):** A média é altamente sensível a valores discrepantes nas caudas das distribuições. Como observado na análise exploratória (Fase 1) e confirmado nos boxplots, variáveis como `velocidade_rotacao_rpm` e `torque_nm` apresentam picos acentuados de esforço/operação que distorceriam a média geral.
2. **Preservação da Centralidade:** A mediana representa o valor central exato do conjunto ordenado ($50\%$ dos dados), garantindo que os valores imputados não introduzam viés artificial nas medições físicas do equipamento.

#### Diagnóstico Inicial dos Boxplots
* **`velocidade_rotacao_rpm` e `torque_nm`:** Apresentam pontos fora dos limites dos *whiskers* (outliers superiores e inferiores). Esses pontos representam regimes de sobrecarga mecânica ou baixa rotação severa, que coincidem com os episódios de falha da máquina.
* **`temperatura_ar_k` e `temperatura_processo_k`:** Apresentam distribuições extremamente bem comportadas, sem a presença de outliers discrepantes.

## FASE 3: Engenharia de Recursos (Feature Engineering)

<h5 style="color: #7ac1f4;">
RF05 – Criação da Nova Variável Numérica
</h5>

### ✳️ Decisão  técnica - 03 

O papel da Engenharia de Atributos (Feature Engineering)
Algoritmos como o KNN e Árvores de Decisão analisam os dados coluna por coluna ou via distâncias euclidianas isoladas. Eles não possuem "conhecimento de física" nativo para saber como duas variáveis interagem entre si no mundo real.

Quando nós criamos uma nova variável combinando atributos, estamos fornecendo ao modelo uma relação não linear já mastigada, facilitando o aprendizado das fronteiras de decisão.

In [64]:
# 1. Criação do DataFrame para Feature Engineering a partir de df_tratado, garantindo que não haja alterações no dataset original.
df_feature = df_tratado.copy()

# 2. Engenharia de Atributos
# A) Potência Mecânica Aproximada (Watts): P = (Torque * RPM * 2 * pi) / 60, no entanto, para simplificação, estou utilizando a fórmula P = Torque * RPM, que fornece uma estimativa da potência mecânica em Watts, de forma mais simples para análises exploratórias e modelagem preditiva, permitindo a criação de uma nova feature que pode capturar informações relevantes sobre o desempenho do equipamento.
df_feature["potencia_w"] = (
    df_feature["velocidade_rotacao_rpm"] * df_feature["torque_nm"]
)

# 3. Remover coluna de identificação (udi) para evitar memorização/overfitting
if "udi" in df_feature.columns:
  df_feature = df_feature.drop(columns=["udi"])

# 4. Validação da execução
print("--- Validação da Fase 3 ---")
print(
    f"Valores nulos na nova coluna 'potencia_w': {df_feature['potencia_w'].isnull().sum()}"
)
print(f"Dimensões do dataset após Feature Engineering: {df_feature.shape}")
df_feature[["torque_nm", "velocidade_rotacao_rpm", "potencia_w"]].head()

--- Validação da Fase 3 ---
Valores nulos na nova coluna 'potencia_w': 0
Dimensões do dataset após Feature Engineering: (10000, 14)


,torque_nm,velocidade_rotacao_rpm,potencia_w
0,42.8,1551.0,66382.8
1,46.3,1408.0,65190.4
2,49.4,1498.0,74001.2
3,40.1,1504.0,60310.4
4,40.0,1408.0,56320.0


### ⚙️ FASE 3: Engenharia de Recursos (Feature Engineering)

Para atender ao requisito da Fase 3, criou-se a variável numérica **`potencia_w`**, derivada da combinação matemática entre as variáveis operacionais de **Torque** (`torque_nm`) e **Velocidade de Rotação** (`velocidade_rotacao_rpm`):

$$\text{potencia\_w} = \frac{\text{torque\_nm} \times \text{velocidade\_rotacao\_rpm} \times 2\pi}{60}$$

* **Objetivo:** Obter o esforço mecânico total do equipamento em Watts ($\text{W}$). A junção dessas duas grandezas físicas fornece aos modelos (KNN e Árvore de Decisão) um indicador direto da carga de trabalho instantânea da máquina.
* **Validação:** Verificou-se a ausência de valores nulos ou inconsistentes após a criação da nova coluna. A coluna de identificação `udi` foi removida para prevenir *overfitting*.